In [2]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [3]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


In [4]:
from pyspark.sql.functions import *

## Biggest Single Number

You are given a DataFrame:

**`my_numbers`**
- `num` (int): A number in the table

**Task:** A "single number" is a number that appears only once in the table. Find the largest single number. If there is no single number, return null/None.

Return a single column `num` with one row containing the result.

### Example
Given:
| num |
|-----|
| 8   |
| 8   |
| 3   |
| 3   |
| 1   |
| 4   |
| 5   |
| 6   |

Single numbers are: 1, 4, 5, 6. The largest is 6.

Expected:
| num |
|-----|
| 6   |

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType

# 1. Start a Spark session
spark = SparkSession.builder.appName("BiggestSingleNumber").getOrCreate()

# 2. Define the data (matches the sample rows)
data = [(8,), (8,), (3,), (3,), (1,), (4,), (5,), (6,)]

# 3. Define the schema (one column 'num' of integer type)
schema = StructType([StructField("num", IntegerType(), True)])

# 4. Create the DataFrame
my_numbers = spark.createDataFrame(data, schema)

my_numbers.show()

+---+
|num|
+---+
|  8|
|  8|
|  3|
|  3|
|  1|
|  4|
|  5|
|  6|
+---+



# Using Spark SQL 

In [7]:
my_numbers.createOrReplaceTempView("numbers")

In [12]:
spark.sql("""with cte as (SELECT num from numbers group by num having count(*) = 1)
    SELECT MAX(num) AS num from cte
""").show()

+---+
|num|
+---+
|  6|
+---+



In [39]:
from pyspark.sql.functions import count, col, max as spark_max

def solution(my_numbers: DataFrame) -> DataFrame:
    singles = (
        my_numbers
        .groupBy("num")
        .agg(count("*").alias("cnt"))
        .filter(col("cnt") == 1)
    )
    
    result = singles.select(spark_max("num").alias("num"))
    return result

solution(my_numbers).show()

+---+
|num|
+---+
|  6|
+---+

